# Not Financial Advice: An Autonomous Investment Research Agent

**AAI-520 Final Team Project, Group 1** · Jackson Kenyon (L1) · Ali Abdul-Hameed (L2) · Nina Zhao (L3) · Ian Schmitt (L4)

GitHub: https://github.com/nazeshinjite/not-financial-advice

Given a stock symbol, our Investment Research Agent plans its research, gathers prices, fundamentals, and news through tools, writes a short research note, grades and revises that note, and stores a lesson that changes how it writes on the next run. It never recommends buying, selling, or holding. We follow the distinction Anthropic draws in *Building Effective Agents* (Schluntz & Zhang, 2024): in a workflow, our code fixes the sequence of LLM calls; in an agent, the model chooses it. Sections 1 to 3 build and demonstrate the three required workflows on their own: prompt chaining (section 1), routing (section 2), and evaluator-optimizer (section 3). Section 4 is the agent. It chooses the chain and the router as tools, passes every note it writes through the evaluator-optimizer, and demonstrates the four required agent functions: planning, dynamic tool use, self-reflection, and learning across runs. Section 5 discusses what the runs show and where the system falls short, and section 6 holds the AI-use disclosure and references.

All market data comes from Yahoo Finance through a committed cache, so every run of this notebook reads the same data. The LLM calls are live, so plans, wording, and scores differ from one run to the next.

## 0. Environment and data

All LLM calls go through `src/llm.py`; all market data through `src/tools.py`, served from the committed cache in `data/cache/`. Symbols: AAPL and NVDA, each run twice by the agent, alternating.

In [ ]:
# Libraries for every section below. Only src/llm.py talks to the model, and only src/tools.py to Yahoo Finance.
import json
import re
import textwrap
from collections import Counter
from html import escape

import matplotlib.pyplot as plt
import pandas as pd
import spacy
from IPython.display import HTML, display

from src import llm, tools
from src.llm import chat, handoff

# Show table cells in full: pandas cuts text at 50 characters by default, which hides the analyses.
pd.set_option("display.max_colwidth", None)

SYMBOLS = ["AAPL", "NVDA"]
print("model:", llm.MODEL)
print("cache:", sorted(p.name for p in tools.CACHE_DIR.iterdir()))


## 1. Prompt chaining

Lane 1 (Jackson Kenyon).

Ingest news, preprocess, classify, extract, summarize. Each stage uses the preceding stage's output.

**Contract:** `run_chain(symbol) -> {"symbol": str, "articles": list[dict], "summary": str}`. The summary is one paragraph synthesized from all labeled articles.

In [ ]:
def ingest(symbol):
    # Fetch cached news articles for the requested company symbol.
    return tools.get_news(symbol)


def preprocess(articles: list[dict], nlp, max_tokens: int = 300) -> list[dict]:
    # Remove duplicate or untitled articles and trim their text for later stages.
    seen_titles = set()
    preprocessed_articles = []
    for article in articles:
        title = (article.get("title") or "").strip()
        summary = (article.get("summary") or "").strip()

        if not title or title in seen_titles:
            continue
        seen_titles.add(title)

        doc = nlp(summary if summary else title)

        token_summary = doc[:max_tokens].text.strip()

        preprocessed_articles.append(
            {
                "title": title,
                "summary": token_summary,
                "publisher": article.get("publisher"),
                "published": article.get("published"),
                "url": article.get("url"),
            }
        )

    return preprocessed_articles


class _ClassificationValidationError(ValueError):
    pass


def classify(article: dict, symbol: str) -> dict:
    sentiment = ["negative", "neutral", "positive"]
    topic = ["earnings", "news", "market", "other"]
    # Classify article sentiment and topic, retrying once if the returned labels are invalid.

    system_prompt = (
        "You are a financial news analyst. Classify the article using its title, summary, and company symbol.\n"
        "The symbol identifies the company the article must be about to count as relevant.\n"
        "Choose the topic that best describes the article's dominant subject, even if it touches multiple areas:\n"
        "- earnings: reported quarterly or annual results, revenue, EPS, margins, earnings growth, financial guidance, or an earnings call.\n"
        "- market: stock-price performance, valuation, trading activity, analyst price targets, sector moves, or broader market/macro effects on the stock.\n"
        "- news: company events that are not primarily earnings or market-price analysis, such as products, leadership, regulation, litigation, partnerships, operations, or corporate announcements.\n"
        "- other: the article is not mainly about the company identified by the symbol, or is unrelated to financial markets.\n"
        "Return a JSON object with exactly two keys:\n"
        '- "sentiment": one of "positive", "negative", or "neutral"\n'
        '- "topic": one of "earnings", "news", "market", or "other"'
    )

    user_prompt = f"Symbol: {symbol}\nTitle: {article['title']}\nSummary: {article['summary']}"

    for attempt in (1, 2):
        output = chat(
            system=system_prompt,
            user=user_prompt,
            json_mode=True,
            temperature=1,
            agent="News Analyst"
        )

        try:
            if not isinstance(output, dict):
                raise _ClassificationValidationError("Classifier reply must be a JSON object.")

            normalized_output = {}
            for label_name, allowed_labels in (("sentiment", sentiment), ("topic", topic)):
                raw_label = output.get(label_name)
                if not isinstance(raw_label, str):
                    raise _ClassificationValidationError(
                        f"Invalid {label_name} label {raw_label!r}; expected one of {allowed_labels}."
                    )

                normalized_label = raw_label.strip().lower()
                if normalized_label not in allowed_labels:
                    raise _ClassificationValidationError(
                        f"Invalid {label_name} label {normalized_label!r}; expected one of {allowed_labels}."
                    )
                normalized_output[label_name] = normalized_label
        except _ClassificationValidationError as error:
            if attempt == 2:
                raise
            print(f"Classifier reply rejected ({error}); asking again.")
            user_prompt = (
                f"{user_prompt}\n\nYour previous reply was rejected: {error}\n"
                "Try again. Return exactly one JSON object with sentiment set to negative, neutral, or positive "
                "and topic set to earnings, news, market, or other."
            )
            continue

        return normalized_output


def extract(article, nlp):
    # Extract named entities from the article title and summary with spaCy.
    title = article.get("title")
    summary = article.get("summary")
    title_doc = nlp(title)
    summary_doc = nlp(summary)

    title_entities = [(ent.text, ent.label_) for ent in title_doc.ents]
    summary_entities = [(ent.text, ent.label_) for ent in summary_doc.ents]

    output = []
    output.append(title_entities)
    output.append(summary_entities)

    return output


def summarize(symbol, articles):
    # Synthesize labeled articles into a concise, source-grounded paragraph for the symbol.
    if not articles:
        return f"No news articles were available for {symbol}."

    article_sections = []
    for index, article in enumerate(articles, start=1):
        title = article.get("title") or "Untitled"
        summary = (article.get("summary") or "").strip() or "No summary provided."
        section = [f"{index}. {title}", f"Summary: {summary}"]
        if article.get("sentiment"):
            section.append(f"sentiment: {article['sentiment']}")
        if article.get("topic"):
            section.append(f"topic: {article['topic']}")
        if article.get("entities"):
            section.append(f"entities: {article['entities']}")
        article_sections.append("\n".join(section))

    system = (
        "You are a careful financial news summarizer. Write exactly one concise paragraph "
        "of no more than five sentences synthesizing the supplied articles about the given symbol. "
        "Use only information in the articles, distinguish reported claims from established facts, "
        "and do not add predictions or investment advice."
    )
    user = f"Symbol: {symbol}\n\nArticles:\n" + "\n\n".join(article_sections)

    return chat(
        system=system,
        user=user,
        max_tokens=500,
        temperature=1,
        agent="News Analyst",
    )


def run_chain(symbol):
    # Run the news-processing stages in order and return labeled articles with a summary.
    # Load spaCy model (disable unnecessary pipeline components for speed)
    nlp = spacy.load("en_core_web_sm", disable=["parser"])

    fetched_articles = ingest(symbol)
    articles = preprocess(fetched_articles, nlp)
    labeled_articles = []
    dropped_count = 0

    for article in articles:
        labeled_article = article.copy()
        labeled_article.update(classify(labeled_article, symbol))
        if labeled_article["topic"] == "other":
            dropped_count += 1
            continue
        labeled_article["entities"] = extract(labeled_article, nlp)
        labeled_articles.append(labeled_article)

    print(f"Dropped {dropped_count} of {len(articles)} articles as other.")
    summary = summarize(symbol, labeled_articles)
    return {
        "symbol": symbol,
        "articles": labeled_articles,
        "summary": summary,
    }


def visualize(result: dict):
    # Display the summary, article details, and sentiment/topic distributions.
    symbol = result.get("symbol", "Unknown")
    summary = result.get("summary") or "No summary available."
    articles = result.get("articles", [])
    table_rows = []

    for article in articles:
        entities = [
            f"{entity_text} ({entity_label})"
            for entity_group in article.get("entities", [])
            for entity_text, entity_label in entity_group
        ]
        values = [
            article.get("title") or "Untitled",
            article.get("sentiment") or "Unlabeled",
            article.get("topic") or "Uncategorized",
            ", ".join(entities) or "None",
        ]
        cells = "".join(f"<td>{escape(str(value))}</td>" for value in values)
        table_rows.append(f"<tr>{cells}</tr>")

    if not table_rows:
        table_rows.append('<tr><td colspan="4">No articles available.</td></tr>')

    table_html = (
        f"<h3>{escape(str(symbol))} summary</h3>"
        f"<p class='chain-summary'>{escape(str(summary))}</p>"
        f"<h3>{escape(str(symbol))} news by article</h3>"
        "<style>"
        ".chain-summary{line-height:1.5;margin:0 0 14px}"
        ".chain-table{border-collapse:collapse;width:100%;table-layout:fixed;font-size:12px}"
        ".chain-table th,.chain-table td{border:1px solid #c9d1d9;padding:6px;"
        "text-align:left;vertical-align:top;overflow-wrap:anywhere}"
        ".chain-table th{background:#eef2f5}"
        ".chain-table tr:nth-child(even){background:#f7f9fa}"
        "</style>"
        '<table class="chain-table"><thead><tr>'
        "<th style='width:36%'>Article</th><th style='width:14%'>Sentiment</th>"
        "<th style='width:14%'>Topic</th><th style='width:36%'>Entities</th>"
        "</tr></thead><tbody>"
        + "".join(table_rows)
        + "</tbody></table>"
    )
    display(HTML(table_html))

    sentiment_labels = ["negative", "neutral", "positive"]
    sentiment_counts = [
        sum(
            1
            for article in articles
            if str(article.get("sentiment", "")).lower() == sentiment
        )
        for sentiment in sentiment_labels
    ]
    topic_labels = ["earnings", "news", "market"]
    topic_counts = [
        sum(
            1
            for article in articles
            if str(article.get("topic", "")).lower() == topic
        )
        for topic in topic_labels
    ]

    figure, (sentiment_axis, topic_axis) = plt.subplots(1, 2, figsize=(12, 4))
    sentiment_bars = sentiment_axis.bar(
        sentiment_labels,
        sentiment_counts,
        color=["#d95f59", "#89939d", "#319b79"],
        width=0.62,
    )
    sentiment_axis.bar_label(sentiment_bars, padding=3)
    sentiment_axis.set_title(f"Sentiment distribution for {symbol}")
    sentiment_axis.set_xlabel("Sentiment")
    sentiment_axis.set_ylabel("Number of articles")
    sentiment_axis.set_ylim(0, max(sentiment_counts, default=0) + 1)

    topic_bars = topic_axis.bar(
        topic_labels,
        topic_counts,
        color=["#4e79a7", "#f28e2b", "#59a14f"],
        width=0.62,
    )
    topic_axis.bar_label(topic_bars, padding=3)
    topic_axis.set_title(f"Topic distribution for {symbol}")
    topic_axis.set_xlabel("Topic")
    topic_axis.set_ylabel("Number of articles")
    topic_axis.set_ylim(0, max(topic_counts, default=0) + 1)

    for chart_axis in (sentiment_axis, topic_axis):
        chart_axis.set_axisbelow(True)
        chart_axis.yaxis.grid(True, color="#dfe3e6", linewidth=0.8)
        chart_axis.spines["top"].set_visible(False)
        chart_axis.spines["right"].set_visible(False)

    figure.tight_layout()
    plt.show()

### Demonstration

The chain runs on NVDA from the committed cache. The output shows how many articles the classifier dropped as not about the company, the one-paragraph summary, each kept article with its sentiment, topic, and entities, and the sentiment and topic counts.

In [ ]:
result = run_chain("NVDA")
visualize(result)

## 2. Routing

Lane 2 (Ali Abdul-Hameed).

An LLM router labels each item as `earnings`, `news`, or `market`, then hands the item to a specialist with a narrower prompt and the right supporting data.

**Contract:** `route(item) -> "earnings" | "news" | "market"`; `analyze_earnings / analyze_news / analyze_market(symbol, context) -> str`; `route_and_analyze(symbol, items) -> [{"item", "route", "analysis"}]`.

The router deliberately reads only the item's `title` and `summary`, so it works with either a raw Yahoo Finance news item or an article that already passed through Lane 1. Earnings analysis adds cached fundamentals, market analysis adds cached price summaries, and news analysis uses the routed article plus the chain summary when one is available.

`route_and_analyze()` accepts either a list of articles or the full dictionary returned by `run_chain()`. Every dispatch is printed with `handoff()` so the notebook visibly demonstrates multi-agent routing.


In [ ]:
# Valid destinations the Router can send financial content to.
VALID_ROUTES = {"earnings", "news", "market"}

# Define the routing rules used to decide which specialist should handle each item.
ROUTER_SYSTEM = """You are the Router in a financial research system.
Choose exactly one specialist for the supplied item:

- earnings: reported quarterly or annual results, revenue, EPS, margins,
  earnings growth, financial guidance, or an earnings call.
- market: stock-price performance, valuation, trading activity, analyst
  price targets, sector moves, or broader market/macro effects on the stock.
- news: company events that are not primarily earnings or market-price
  analysis, such as products, leadership, regulation, litigation,
  partnerships, operations, or corporate announcements.

Classify the dominant subject, even when an item touches more than one area.
Return JSON only in exactly this shape:
{"route": "earnings"}
The value must be one of: earnings, news, market.
"""


def _context_text(context, max_chars=6000):
    """Convert specialist context to compact prompt text."""
    # Specialist context may be plain text or structured data from another lane.
    if isinstance(context, str):
        text = context
    else:
        text = json.dumps(context, ensure_ascii=False, default=str, indent=2)

    # Limit prompt size so large upstream results do not overwhelm the model.
    return text[:max_chars]


def route(item):
    """Return the specialist label for one raw or chain-processed news item."""

    # Routing expects a dictionary containing at least a title or summary.
    if not isinstance(item, dict):
        raise TypeError("route(item) expects a dictionary")

    title = (item.get("title") or "").strip()
    summary = (item.get("summary") or "").strip()

    if not title and not summary:
        raise ValueError("route(item) needs a title, summary, or both")

    # Only title and summary are used so both raw Yahoo items and Lane 1 output work.
    user = f"Title: {title or '(none)'}\nSummary: {summary or '(none)'}"

    # Ask the Router agent to select the most appropriate specialist.
    result = chat(
        system=ROUTER_SYSTEM,
        user=user,
        json_mode=True,
        max_tokens=100,
        temperature=1.0,
        agent="Router",
    )

    label = str(result.get("route", "")).strip().lower()

    # Fall back to News instead of stopping the full agent on an unexpected label.
    if label not in VALID_ROUTES:
        print(f"Router returned {label!r}; routing to news")
        label = "news"

    return label


def analyze_earnings(symbol, context):
    """Analyze routed earnings content using cached company fundamentals."""

    # Add Yahoo Finance fundamentals so the specialist has financial context.
    fundamentals = tools.get_fundamentals(symbol)

    system = """You are the Earnings Specialist in a financial research system.
Analyze the routed content using only the supplied context and fundamentals.
Focus on reported financial performance, growth, margins, earnings, or guidance
when relevant. Clearly separate article claims from the Yahoo Finance data.
Write 2-4 concise sentences. Do not predict the stock price and do not give
buy, sell, or hold advice.
"""

    user = (
        f"Symbol: {symbol}\n\n"
        f"Routed context:\n{_context_text(context)}\n\n"
        "Cached Yahoo Finance fundamentals:\n"
        f"{json.dumps(fundamentals, indent=2, default=str)}"
    )

    return chat(
        system=system,
        user=user,
        max_tokens=1500,
        temperature=1.0,
        agent="Earnings Specialist",
    )


def analyze_news(symbol, context):
    """Analyze routed company news; chain summary is used when supplied."""

    # News analysis relies on the routed article and optional summary from Lane 1.
    system = """You are the News Specialist in a financial research system.
Analyze the supplied company-news context. Explain what happened and why it
could matter to the company, while staying grounded only in the supplied text.
If both an individual item and a chain summary are present, use the chain
summary as broader context without inventing facts. Write 2-4 concise
sentences. Do not predict the stock price and do not give investment advice.
"""

    user = f"Symbol: {symbol}\n\nNews context:\n{_context_text(context)}"

    return chat(
        system=system,
        user=user,
        max_tokens=1500,
        temperature=1.0,
        agent="News Specialist",
    )


def analyze_market(symbol, context):
    """Analyze routed market content using cached price-summary data."""

    # Pull cached price data and remove the long daily arrays before prompting.
    prices = tools.get_prices(symbol)
    market_data = {
        key: value
        for key, value in prices.items()
        if key not in {"dates", "close"}
    }

    system = """You are the Market Specialist in a financial research system.
Analyze the routed content using only the supplied context and cached market
data. Focus on observable price performance, range, trading context, valuation
or broader market effects when relevant. Do not invent catalysts or forecasts.
Write 2-4 concise sentences and do not give buy, sell, or hold advice.
"""

    user = (
        f"Symbol: {symbol}\n\n"
        f"Routed context:\n{_context_text(context)}\n\n"
        "Cached Yahoo Finance market summary:\n"
        f"{json.dumps(market_data, indent=2, default=str)}"
    )

    return chat(
        system=system,
        user=user,
        max_tokens=1500,
        temperature=1.0,
        agent="Market Specialist",
    )


# Map each router label to the specialist function that performs the analysis.
SPECIALISTS = {
    "earnings": analyze_earnings,
    "news": analyze_news,
    "market": analyze_market,
}

# Keep the displayed agent names consistent with the shared project contract.
SPECIALIST_NAMES = {
    "earnings": "Earnings Specialist",
    "news": "News Specialist",
    "market": "Market Specialist",
}


def route_and_analyze(symbol, items):
    """Route each item, dispatch it, and return the routing decisions."""

    chain_summary = None

    # Accept either the full Lane 1 run_chain() result, one item, or a list of items.
    if isinstance(items, dict) and "articles" in items:
        chain_summary = items.get("summary")
        items = items.get("articles", [])
    elif isinstance(items, dict):
        items = [items]
    else:
        items = list(items)

    results = []

    # Route each item, show the handoff, then run the selected specialist.
    for item in items:
        label = route(item)
        specialist_name = SPECIALIST_NAMES[label]

        handoff(
            "Router",
            specialist_name,
            item.get("title") or item.get("summary") or str(item),
        )

        specialist_context = item

        # Give the News Specialist the broader Lane 1 summary when available.
        if label == "news" and chain_summary:
            specialist_context = {
                "item": item,
                "chain_summary": chain_summary,
            }

        analysis = SPECIALISTS[label](symbol, specialist_context)

        # Keep the original item, routing decision, and specialist output together.
        results.append(
            {
                "item": item,
                "route": label,
                "analysis": analysis,
            }
        )

    return results

### Demonstration

The demonstration builds a three-item mixed batch from the committed AAPL cache: one fundamentals/earnings item, one real Yahoo Finance company-news item, and one market-performance item. The router must choose a specialist for each item, and the specialist output is shown beside the decision.

The bar chart makes the routing behavior visible in the final HTML submission. Because all supporting Yahoo Finance data comes from the committed cache, rerunning the notebook does not change the evidence used by the specialists.


In [ ]:
symbol = "AAPL"

# Load cached data used to build one example for each routing category.
fundamentals = tools.get_fundamentals(symbol)
prices = tools.get_prices(symbol)
company_news = tools.get_news(symbol)[0]

# Create a mixed batch that should exercise Earnings, News, and Market routing.
demo_items = [
    {
        "title": "Apple earnings and fundamentals snapshot",
        "summary": (
            f"Revenue is {fundamentals.get('revenue_b')} billion "
            f"{fundamentals.get('currency')}; profit margin is "
            f"{fundamentals.get('profit_margin_pct')}% and earnings "
            f"growth is {fundamentals.get('earnings_growth_pct')}%."
        ),
    },
    company_news,
    {
        "title": "Apple six-month market performance",
        "summary": (
            f"Over {prices.get('period')}, AAPL changed "
            f"{prices.get('change_pct')}%. The latest close was "
            f"{prices.get('last_close')}, with a high of "
            f"{prices.get('high')} and low of {prices.get('low')}."
        ),
    },
]

# Run the mixed batch through the Router and its selected specialists.
routed = route_and_analyze(symbol, demo_items)

# Display the routing decision and a short excerpt of each specialist response.
routing_table = pd.DataFrame(
    {
        "item": [row["item"].get("title", "") for row in routed],
        "route": [row["route"] for row in routed],
        "analysis excerpt": [row["analysis"][:180] for row in routed],
    }
)
display(routing_table)

# Count how many demo items were sent to each specialist.
route_order = ["earnings", "news", "market"]
route_counts = (
    routing_table["route"]
    .value_counts()
    .reindex(route_order, fill_value=0)
)

# Visualize the routing decisions for the final notebook demonstration.
plt.figure(figsize=(6, 4))
plt.bar(route_counts.index, route_counts.values)
plt.title("Routing Decisions for Mixed AAPL Batch")
plt.xlabel("Specialist route")
plt.ylabel("Number of items")
plt.tight_layout()
plt.show()

## 3. Evaluator-optimizer

Lane 3 (Nina Zhao).

Generate an analysis, score it against explicit criteria, refine it from the feedback, at most two rounds.

**Contract:** `generate_analysis(symbol, context) -> str`; `evaluate(text, criteria) -> {"score": 1-10, "pass": bool, "feedback": list[str]}`, each feedback item naming the criterion it failed; `refine(text, feedback) -> str`; `evaluator_optimizer(generate, criteria, max_rounds=2) -> {"final": str, "history": [{"round", "score", "pass", "feedback"}]}`.
`generate` is a zero-argument callable so the agent can pass its own report writer.

**Extra keys beyond the contract** (callers may ignore them): `evaluate()` also returns `checks`, one `{"number", "reason", "met"}` verdict per criterion; `evaluator_optimizer()` also returns `drafts`, every version in order. When a revision scores lower than an earlier draft, the earlier draft is kept as `final` and its grade is repeated as the last `history` entry, marked `"kept": true`, so `history[-1]` is always the grade of `final`.

### How the loop works

Two agents take turns. The **Writer** drafts an analysis. The **Critic** checks the draft against a numbered checklist and the data it was written from, one criterion at a time, and says which criteria failed and why. The Writer then revises the draft to fix exactly those points. The Critic grades the revision, and the loop stops when every criterion is met or after two revisions.

Design choices:

- **Criteria are checkable, not vague.** Each one names something the Critic can look for in the text and verify in the data ("states the profit margin percent"), not a judgment like "is insightful".
- **The criteria carry the reference data.** `make_criteria` appends the data the analysis was written from, so the Critic checks that a figure is *correct*, not only that one is present. Lane 4 builds its criteria the same way.
- **The score is computed, not guessed.** The Critic marks each criterion met or not met, and the score is the share met, scaled to 10. In team testing, a model asked for a 1-to-10 score directly gave the same text scores about two points apart; a per-criterion yes or no is more stable, and the feedback names exactly what to fix.
- **The loop keeps the best draft, and always ends with its grade.** A revision can score lower than the draft it revised, so `final` is the best-scoring draft (the latest one on a tie), and `history[-1]` is always the grade of `final`. `history[0]` records what the first draft got wrong. The agent in Lane 4 reads both.
- **The loop is bounded**: at most two revisions (five LLM calls when no reply needs a retry), so it always terminates.

### Reference data and criteria

In [ ]:
# The data one analysis is written from: the six-month price summary, the fundamentals, and the
# recent headlines, all from the committed cache, as one JSON string. The Writer reads it and the
# Critic checks every figure against it.
def build_context(symbol):
    prices = tools.get_prices(symbol)
    fundamentals = tools.get_fundamentals(symbol)
    view = {
        # The 127 daily closes are dropped; the summary figures are what an analysis quotes.
        "prices_6mo": {k: v for k, v in prices.items() if k not in ("dates", "close")},
        # Two decimals: Yahoo's P/E arrives as 38.591274, which nobody quotes.
        "fundamentals": {k: round(v, 2) if isinstance(v, float) else v for k, v in fundamentals.items()},
        "headlines": [article["title"] for article in tools.get_news(symbol)],
    }
    return json.dumps(view, indent=1)


# The Critic's checklist for this lane's demonstration. Every criterion can be checked in the text
# and verified against the reference data. Lane 4 passes its own list in the same numbered format.
CRITERIA = """\
1. States the share price change over the six months as a percent, and the last closing price.
2. States total revenue with its unit, and the revenue growth percent.
3. States the profit margin percent.
4. States the trailing or forward P/E ratio and says in one phrase what it suggests about valuation.
5. Refers to at least one specific headline from the reference data.
6. Names at least two distinct risks, each supported by a figure or a headline in the reference data.
7. Every figure in the text matches the reference data (rounding and unit changes such as 4905.54 billion as 4.91 trillion are fine), and the text states no fact or comparison the data does not contain.
8. Makes no recommendation to buy, sell, or hold (saying that no recommendation is made is fine).
9. Is written in plain words for a reader: no data field names such as revenue_b or change_pct.
"""


# The checklist plus the data the text was written from, so the Critic can check that a figure is
# correct, not only that one is present.
def make_criteria(context, criteria=CRITERIA):
    return f"{criteria}\nReference data:\n{context}"

### Writer: the first draft

`generate_analysis` is a deliberately plain prompt: it does not show the Writer the checklist. A first draft written this way usually misses a few criteria, which gives the Critic something real to catch and the loop something to improve. Inside the agent (Lane 4), the Writer's own report function takes this place.

In [ ]:
WRITER = """\
You are the Writer on an investment research team. Write a short research analysis of the stock
below, using only the data given. Do not recommend buying, selling, or holding.
"""


# One plain chat() call. Exists so the loop can be demonstrated without the agent (section 4).
def generate_analysis(symbol, context):
    return chat(WRITER, f"Stock: {symbol}\n\nData:\n{context}", max_tokens=1500, agent="Writer")

### Critic: grading against the criteria

The Critic replies in JSON with one verdict per criterion. The code then checks the reply before trusting it: it must contain a verdict for every numbered criterion, each with a true or false `met`. A reply that fails the check, or a call that fails (for example a reply cut off at the token limit, which happened about once in 100 calls in team testing), is retried once. The score, pass flag, and feedback list are computed from the verdicts in code, so the result always has all three keys.

In [ ]:
CRITIC = """\
You are the Critic on an investment research team. Judge the text against each numbered criterion
separately, using the reference data to check every figure. A criterion is met only if the text
clearly satisfies it.
Reply with JSON only, in exactly this shape:
{"criteria": [{"number": 1, "reason": "<one sentence>", "met": true}, ...]}
Include every criterion, in order. Write the reason first, then set "met" (true or false) to agree
with it. For a criterion that is not met, the reason says what is missing or wrong, specifically
enough for a writer to fix it.
"""


# How many numbered criteria the checklist has (the lines before the reference data that start "N.").
def count_criteria(criteria):
    checklist = criteria.split("Reference data:")[0]
    return len(re.findall(r"^\s*\d+\.", checklist, flags=re.MULTILINE))


# The Critic's verdicts, checked: one per criterion, each with a number and a true/false "met".
# Raises ValueError on a reply that does not have that shape.
def parse_verdicts(reply, expected):
    verdicts = reply.get("criteria")
    if not isinstance(verdicts, list) or len(verdicts) != expected:
        raise ValueError(f"expected {expected} verdicts, got {verdicts!r:.200}")
    checked = []
    for verdict in verdicts:
        met = verdict.get("met") if isinstance(verdict, dict) else None
        if isinstance(met, str) and met.lower() in ("true", "false"):
            met = met.lower() == "true"  # a model occasionally writes "true" as a string
        if not isinstance(met, bool) or "number" not in verdict:
            raise ValueError(f"malformed verdict: {verdict!r:.200}")
        checked.append({"number": verdict["number"], "met": met, "reason": str(verdict.get("reason", ""))})
    return checked


# One Critic call (retried once on a bad reply) -> {"score", "pass", "feedback", "checks"}. The score
# is the share of criteria met, out of 10; feedback lists only the failed criteria, each named.
def evaluate(text, criteria):
    expected = count_criteria(criteria)
    user = f"CRITERIA:\n{criteria}\n\nTEXT TO JUDGE:\n{text}"
    for attempt in (1, 2):
        try:
            reply = chat(CRITIC, user, json_mode=True, max_tokens=1500, agent="Critic")
            checks = parse_verdicts(reply, expected)
            break
        except ValueError as error:  # truncated, not JSON, or the wrong shape
            if attempt == 2:
                raise
            print(f"Critic reply rejected ({error}); asking again.")
    failed = [f"Criterion {c['number']}: {c['reason']}" for c in checks if not c["met"]]
    met = len(checks) - len(failed)
    return {"score": round(10 * met / len(checks)), "pass": not failed, "feedback": failed, "checks": checks}

### Writer: revising from the feedback

`refine` gives the Writer the Critic's feedback and the draft, and asks it to fix those points and change nothing else. The loop also passes the criteria and reference data, so a figure the Writer adds comes from the data rather than from memory. `reference` is optional, so `refine(text, feedback)` still matches the contract.

In [ ]:
REVISER = """\
You are the Writer on an investment research team, revising your analysis after the Critic's
review. Fix every point in the feedback, and take any figure you add or correct from the reference
data only. Keep everything the feedback does not mention as it is. Reply with the revised analysis
only, with no commentary on what changed.
"""


# feedback is the list from evaluate(); reference is the criteria and data, when the caller has them.
def refine(text, feedback, reference=""):
    notes = "\n".join(f"- {item}" for item in feedback)
    data = f"\n\nCRITERIA AND REFERENCE DATA:\n{reference}" if reference else ""
    user = f"FEEDBACK:\n{notes}\n\nANALYSIS TO REVISE:\n{text}{data}"
    return chat(REVISER, user, max_tokens=1500, agent="Writer")

### The loop

In [ ]:
# generate() -> evaluate -> refine -> evaluate, stopping on a pass or after max_rounds revisions.
# Returns the best-scoring draft as "final"; history[-1] is always the grade of "final", and
# history[0] records what the first draft got wrong. "drafts" keeps every version, in order.
def evaluator_optimizer(generate, criteria, max_rounds=2):
    text = generate()
    drafts = [text]
    handoff("Writer", "Critic", f"first draft, {len(text.split())} words")
    check = evaluate(text, criteria)
    history = [{"round": 0, **check}]
    for round_number in range(1, max_rounds + 1):
        if check["pass"]:
            break
        handoff("Critic", "Writer", f"score {check['score']}/10, {len(check['feedback'])} criteria to fix")
        text = refine(text, check["feedback"], reference=criteria)
        drafts.append(text)
        handoff("Writer", "Critic", f"revision {round_number}, {len(text.split())} words")
        check = evaluate(text, criteria)
        history.append({"round": round_number, **check})
    # Keep the best-scoring draft (the latest on a tie): a revision can score lower than the draft it
    # revised. When the best is not the last, its grade is repeated at the end, marked "kept", so
    # history[-1] is always the grade of "final". No extra LLM call is made.
    best = max(range(len(history)), key=lambda i: (history[i]["score"], i))
    if best != len(history) - 1:
        history.append({**history[best], "kept": True})
        handoff("Critic", "Writer", f"keeping round {best} ({history[best]['score']}/10); later revisions scored lower")
    return {"final": drafts[best], "history": history, "drafts": drafts}

### Demonstration

Three drafts go through the loop. The first is a full analysis of AAPL from the plain Writer prompt: when it meets every criterion, the loop stops after one grading, and when it does not, it shows how far two revisions take a draft that is already close. The other two are deliberately thin two-sentence drafts of NVDA and AAPL, which miss most of the checklist and show the loop finding the gaps and the Writer filling them. Each line below is one handoff between the Writer and the Critic.

In [ ]:
# A thin first draft: two sentences, so it misses most criteria on purpose.
def thin_draft(symbol, context):
    prompt = f"In two sentences, describe {symbol} using this data:\n{context}"
    return chat(WRITER, prompt, max_tokens=300, agent="Writer")


cases = [
    ("AAPL, full draft", "AAPL", generate_analysis),
    ("NVDA, thin draft", "NVDA", thin_draft),
    ("AAPL, thin draft", "AAPL", thin_draft),
]
demo_start = len(llm.CALL_LOG)  # count only this demonstration's calls, even after a re-run
runs = {}
for label, symbol, writer in cases:
    print(f"\n=== {label} ===")
    context = build_context(symbol)
    runs[label] = evaluator_optimizer(
        lambda s=symbol, c=context, w=writer: w(s, c), make_criteria(context)
    )
    final = runs[label]["history"][-1]
    handoff("Critic", "Coordinator", f"final score {final['score']}/10, pass={final['pass']}")

### Score per round

One row per Critic grading. `failed` lists the numbers of the criteria the draft did not meet. When a revision scored lower than an earlier draft, a last row marks the earlier draft kept as final.

In [ ]:
rows = []
for label, run in runs.items():
    for h in run["history"]:
        failed = [c["number"] for c in h["checks"] if not c["met"]]
        rows.append({"draft": label, "round": h["round"], "score": h["score"], "pass": h["pass"],
                     "failed": ", ".join(str(n) for n in failed) or "none",
                     "note": "best draft, kept as final" if h.get("kept") else ""})
display(pd.DataFrame(rows))

fig, ax = plt.subplots(figsize=(7, 4))
for label, run in runs.items():
    graded = [h for h in run["history"] if not h.get("kept")]  # one point per grading
    ax.plot([h["round"] for h in graded], [h["score"] for h in graded], marker="o", label=label)
ax.axhline(10, color="gray", linestyle=":", linewidth=1)
ax.set_xticks([0, 1, 2], ["first draft", "revision 1", "revision 2"])
ax.set(title="Critic score per round", ylabel="score (criteria met, out of 10)", ylim=(0, 10.5))
ax.legend()
plt.show()

### Before and after

The draft whose score rose the most. First, each criterion's verdict on the first draft and on the final text; then the Critic's feedback on the first draft; then the two texts, so the gap the feedback named can be seen filled in the revision.

In [ ]:
def gain(label):
    history = runs[label]["history"]
    return history[-1]["score"] - history[0]["score"]


best = max(runs, key=gain)
run = runs[best]
first, last = run["history"][0], run["history"][-1]
criteria_lines = [line for line in CRITERIA.splitlines() if re.match(r"\s*\d+\.", line)]
mark = {True: "met", False: "NOT met"}
display(pd.DataFrame({
    "criterion": [textwrap.shorten(line, 70) for line in criteria_lines],
    "first draft": [mark[c["met"]] for c in first["checks"]],
    "final": [mark[c["met"]] for c in last["checks"]],
}))


def show(title, text):
    print(f"\n{title}\n" + "-" * len(title))
    for paragraph in text.split("\n"):
        print(textwrap.fill(paragraph, 100) if paragraph.strip() else "")


print(f"Draft: {best}. Score {first['score']}/10 -> {last['score']}/10 "
      f"after {len(run['drafts']) - 1} revision(s).")
show("What the Critic said about the first draft", "\n".join(first["feedback"]) or "no failed criteria")
show("First draft", run["drafts"][0])
show("Final text", run["final"])

### LLM calls by agent

Every call goes through `chat()`, which records the agent that made it. One loop costs at most five calls when no reply needs a retry: one first draft, and up to two revisions with a grading after each draft.

In [ ]:
calls = pd.DataFrame(llm.CALL_LOG[demo_start:])
calls["tokens"] = calls["prompt_tokens"].fillna(0) + calls["completion_tokens"].fillna(0)
display(calls.groupby("agent").agg(calls=("agent", "size"), tokens=("tokens", "sum"),
                                   seconds=("seconds", "sum")).astype(int))

## 4. Investment Research Agent

Lane 4 (Ian Schmitt).

### Agent Design and Workflows

Given a stock symbol, the agent decides which research steps to take, takes them, writes a research note of 150 to 250 words from what it found, grades and revises the note, and stores a lesson that changes how it writes the next time. Sections 1 to 3 are workflows, with the sequence of calls fixed in our code. Here the Coordinator chooses the sequence: the chain and the router are among the tools it can call, and every note then goes through the evaluator-optimizer loop.

One research cycle:

```
 1 PLAN      Coordinator picks steps -> [{"tool", "args", "why"}]
                 |
 2 EXECUTE   for each step, by name:
               get_prices / get_fundamentals                (src/tools.py, cached)
               run_chain          -> News Analyst            (section 1)
               route_and_analyze  -> Router + Specialists    (section 2)
             every step logged; a failed step is recorded, not fatal
                 |
            memory: ["lesson", ...]  <------------------------------------------------+
                 |                                                                    |
 3 WRITE     Writer drafts from the data and memory; Critic   (section 3, <= 2 rounds)|
             grades each criterion; Writer revises                                    |
                 |                                                                    |
 4 REFLECT   Reflector reads the run, above all the first draft's feedback            |
               -> {"score", "lesson"}                                                 |
                 |                                                                    |
 5 REMEMBER  a new lesson is added  -------------------------------------------------+
```

The four required agent functions map onto this cycle: planning is step 1, dynamic tool use is step 2, self-reflection is steps 3 and 4, and learning across runs is step 5 feeding the next run's step 3.

The system is one agent loop staffed by named roles. Every LLM call names its role through `chat(..., agent="Name")`, which records it in `llm.CALL_LOG`, and every pass of work from one role to another prints one line through `handoff(sender, receiver, content)`. The output below therefore reads as a transcript of who handed what to whom, as in the Module 7 lab's multi-agent team, without an agent framework.

| Agent | Section | Job |
|---|---|---|
| Coordinator | 4 | Plans the research and dispatches each step |
| News Analyst | 1 | Runs the prompt chain over recent news |
| Router | 2 | Labels each article earnings, news, or market |
| Earnings, News, and Market Specialists | 2 | Analyze the articles routed to them, each with its own data |
| Writer | 4 (drafts), 3 (revises) | Writes and revises the research note |
| Critic | 3 | Grades a note against explicit criteria and lists what failed |
| Reflector | 4 | Turns a finished run into a lesson for memory |

Two small modules sit under every section. `src/llm.py` holds `chat()`, the only path to the model (DeepSeek V4.1 Flash through the Nous Portal gateway; DeepSeek-AI, 2026). It returns text or a parsed JSON object, and it raises on a truncated or empty reply instead of passing a broken answer downstream. `src/tools.py` holds the Yahoo Finance tools, built on `yfinance` (Aroussi, n.d.). Each caches its response as JSON under `data/cache/`, and the cache is committed. Section 1's preprocessing and entity extraction use spaCy's small English model (Honnibal et al., 2020). We used no agent framework: the plan is a list, the dispatch is a dictionary lookup, and memory is a list, so every decision is visible in this notebook.

**Contract:** `ResearchAgent.plan(symbol) -> [{"tool", "args", "why"}]` (max 8 steps); `run(symbol) -> dict` (dispatches by name over `get_prices`, `get_fundamentals`, `run_chain`, and `route_and_analyze`, logs every call, writes the report through Lane 3's loop, with memory in the Writer's prompt); `reflect(symbol, run_result) -> {"score", "lesson"}` (takes `run()`'s output, the plan, tool log, report, and score history; takes the score and feedback from the writing loop's last grade of the final note and writes a lesson for the Writer on what the first draft missed, or null when it missed nothing); `remember(lesson)` (adds the lesson, if any, to `self.lessons`, a list that starts empty on every notebook run).

**Demonstration:** AAPL and NVDA twice each, interleaved (AAPL, NVDA, AAPL, NVDA), so NVDA's first run reads AAPL's lessons and each symbol's second run reads every lesson learned before it. Every run with lessons to read is paired with a twin: the same run by a new agent with empty memory, so a difference in plan, note, or score comes from memory (one pair per run, so it suggests rather than proves). Every call samples at the model card's recommended temperature, 1.0.

In [ ]:
# Every step the planner can choose, by name. The executor looks the name up here and calls it.
# Built once when this cell runs, so the lane functions must be defined (or imported) above it.
# get_news is not a step: run_chain fetches the same articles itself, so planning both read the news twice.
STEPS = {
    "get_prices": tools.get_prices,
    "get_fundamentals": tools.get_fundamentals,
    "run_chain": run_chain,
    "route_and_analyze": route_and_analyze,
}

# What the Coordinator reads when planning: what each step returns, what it costs, and its args.
# The args shown are the cached ones; any other value fetches live from Yahoo.
MENU = """\
get_prices  args {"period": "6mo"}  Daily closes plus last_close, change_pct, high, low, avg_volume. No LLM calls.
get_fundamentals  args {}  Name, sector, market cap and revenue (billions), P/E, revenue and earnings growth and
    profit margin (percent), 52-week range. No LLM calls.
run_chain  args {}  Fetches the recent news itself, labels each article's sentiment and topic, extracts
    entities, and writes one summary. One LLM call per article.
route_and_analyze  args {}  Sends each article to an earnings, news, or market specialist for a short
    analysis. Two LLM calls per article. Uses run_chain's labeled articles and summary if run_chain ran
    earlier in the plan, otherwise the raw news.
"""


In [ ]:
# The figures a note would cite, from whichever steps ran, as one JSON string. The Writer, the Critic's
# reference data, and the Reflector all read this view, so the numbers the Writer quotes are the numbers
# the Critic checks. The 127 daily closes are dropped to keep prompts small.
def brief(results):
    view = {}
    if "get_prices" in results:
        view["prices"] = {k: v for k, v in results["get_prices"].items() if k not in ("dates", "close")}
    if "get_fundamentals" in results:
        # Two decimals: Yahoo's P/E arrives as 28.099874, and the Writer quotes figures exactly.
        view["fundamentals"] = {k: round(v, 2) if isinstance(v, float) else v
                                for k, v in results["get_fundamentals"].items()}
    if "run_chain" in results:
        view["headlines"] = [article["title"] for article in results["run_chain"]["articles"]]
        view["news_summary"] = results["run_chain"]["summary"]
        # All three labels, zeros included: Counter drops a label with no articles, and a Critic that
        # cannot find "0 positive" in the data fails a note for stating it.
        counts = Counter(a["sentiment"] for a in results["run_chain"]["articles"])
        view["sentiment_counts"] = {label: counts[label] for label in ("positive", "negative", "neutral")}
        topics = Counter(a["topic"] for a in results["run_chain"]["articles"])
        view["topic_counts"] = {topic: topics[topic] for topic in ("earnings", "news", "market")}
    if "route_and_analyze" in results:
        view["specialist_analyses"] = [
            {"route": row["route"], "title": row["item"]["title"], "analysis": row["analysis"]}
            for row in results["route_and_analyze"]
        ]
    return json.dumps(view, indent=1)


NOTE_WRITER = """\
You are the Writer on an investment research team. Write a research note of 150 to 250 words on the
stock below, using only the data given; if the data does not cover something, leave it out.
Write figures in plain words with their units ("revenue of 416.2 billion USD", "down 4.3 percent over
six months"), never as data field names. Quote figures exactly as given and compute no new ones.
When you discuss risks, take them from the specialist analyses where they name one, and cite the
headline or figure each comes from.
Do not recommend buying, selling, or holding. If notes from earlier runs are given, follow them.
Write in American English.
"""


# The Writer's draft. Passed to Lane 3's loop as a zero-argument generate, so the loop never needs
# to know where the text comes from. recall is the agent's lessons as text, empty when it has none:
# a failure in the writing can only be fixed by telling the Writer. 1500 tokens: with the real
# chain's material a note ran past 800, and a cut-off note is worth nothing.
def write_report(symbol, results, recall=""):
    memory = f"\n\nNotes from earlier runs:\n{recall}" if recall else ""
    return chat(NOTE_WRITER, f"Stock: {symbol}\n\nData:\n{brief(results)}{memory}", max_tokens=1500, agent="Writer")


# What the Critic grades every note against. Fixed, whatever the plan gathered: a plan that skipped
# a needed step produces a note that fails a criterion, and that failure is what reflection learns from.
NOTE_CRITERIA = """\
1. States the share price change over the period, with its percent figure.
2. Cites at least one fundamental (revenue, market cap, P/E, growth, or margin) with its unit.
3. States the sentiment of recent news as counts of positive, negative, and neutral articles.
4. Names at least one specific risk, supported by a figure or a headline in the reference data.
5. Makes no recommendation to buy, sell, or hold (saying that no recommendation is made is fine).
6. Every figure in the note matches the reference data (rounding is fine).
7. Names the most common news topic among the labeled articles, with its count.
"""


# The criteria plus the data the note was written from, so the Critic can check that a figure is
# correct, not only that one is present.
def report_criteria(results):
    return f"{NOTE_CRITERIA}\nReference data:\n{brief(results)}"


# Run one plan step by name. route_and_analyze is the one step that needs another step's output: it
# takes run_chain's whole result (the labeled articles, plus the summary its News Specialist reads) if
# the chain ran, otherwise the raw news. Everything else takes the symbol plus the plan's args.
def call_step(name, args, symbol, results):
    if name not in STEPS:
        raise ValueError(f"unknown step {name!r}; the menu has {', '.join(STEPS)}")
    if name == "route_and_analyze":
        return STEPS[name](symbol, results.get("run_chain") or tools.get_news(symbol))
    return STEPS[name](symbol, **args)


In [ ]:
# The Coordinator's instructions: the goal and the menu. It does not read memory; what a good note needs
# is learned by the Writer, from the Critic's feedback on its first drafts.
PLANNER = f"""\
You are the Coordinator of an investment research team. Plan the research for one stock symbol.
The team will write a short research note on the stock from the data your plan gathers.
The note states facts and never recommends buying, selling, or holding.

Steps you can run:
{MENU}
Rules:
- Choose only the steps the note needs. LLM calls cost time; do not run a step the note will not use.
- If you run route_and_analyze, run run_chain before it.
- At most 8 steps.
- Only the steps above exist.

Reply as JSON: {{"steps": [{{"tool": "<step name>", "args": {{...}}, "why": "<one sentence>"}}]}}
"""


# The Reflector's instructions. It reads the record of one run and writes the lesson the Writer reads
# next time, on any symbol. It learns from the first draft: the writing loop usually rescues a weak
# draft, and a Reflector that saw only the final grade had nothing to learn while the next first draft
# repeated the same mistakes. In testing, memory written after a clean run and rules about what to avoid
# made later notes worse; the rules below stop both. Figures stay out of memory: the chain's labels vary
# between runs, and a note quoting last run's count contradicted the next run's data. Memory is lessons
# only: per-symbol notes were dropped after two full runs stored none, every miss being one any note can make.
REFLECTOR = """\
You are the Reflector on an investment research team. You read the record of one research run and
write memory that will make the Writer's next first draft right without revision.
- lesson: one sentence for the criteria the first draft failed (first_draft_feedback) and what the
  Writer must do to meet them, or null if the first draft drew no feedback. The Writer reads it on
  every symbol. Do not repeat a lesson that is already stored.
- Write instructions, not figures: say what to state and which part of the data holds it, never the
  values themselves. The next run's data may differ.
- Never ask for less than a criterion requires; for example, naming a risk takes a judgment, so never
  tell the Writer to avoid judgments. Say what to do, not what to avoid.
Base the lesson only on the run record; do not invent failures it does not show.
Reply as JSON: {"lesson": "..." or null}
"""


class ResearchAgent:
    # Memory lives on the agent and starts empty, so every notebook run is a cold start.
    # Each lesson is one instruction for the Writer that applies to any symbol.
    def __init__(self):
        self.lessons = []

    # Memory as text for the Writer: every lesson, one per line; empty when there are none.
    def recall(self):
        return "\n".join(f"- {lesson}" for lesson in self.lessons)

    # Ask the Coordinator for an ordered list of steps {"tool", "args", "why"}.
    def plan(self, symbol):
        # 1500 tokens, like the Writer and Critic: twice the planner used all 800 and returned nothing
        # visible, likely hidden reasoning a backend did not switch off. The demo also retries a failed run.
        reply = chat(PLANNER, f"Symbol: {symbol}", json_mode=True, max_tokens=1500, agent="Coordinator")
        # JSON mode returns an object, so the list arrives wrapped in "steps". Unknown tool names are
        # kept: the executor logs and skips them, so a planning mistake stays visible.
        return [
            {"tool": step.get("tool"), "args": step.get("args") or {}, "why": step.get("why", "")}
            for step in reply.get("steps", [])[:8]
        ]

    # Plan, execute the plan step by step, then write the note through Lane 3's loop. Returns
    # everything reflect() needs. A failed step is logged and skipped, not raised: a run that loses
    # one source should still produce a note, and the failure is evidence for reflection.
    def run(self, symbol):
        plan = self.plan(symbol)
        results, log = {}, []
        for step in plan:
            handoff("Coordinator", step["tool"], step["why"])
            try:
                output = call_step(step["tool"], step["args"], symbol, results)
            except Exception as error:  # noqa: BLE001  any failure, from bad args to a timeout, is logged
                log.append({**step, "ok": False, "result": f"{type(error).__name__}: {error}"})
                print(f"    failed: {log[-1]['result']}")
                continue
            results[step["tool"]] = output
            log.append({**step, "ok": True, "result": str(output)[:80]})

        criteria = report_criteria(results)
        handoff("Coordinator", "Writer", f"{symbol}: data from {', '.join(results)}")
        loop = evaluator_optimizer(lambda: write_report(symbol, results, self.recall()), criteria)
        handoff("Critic", "Coordinator", f"final score {loop['history'][-1]['score']}/10")
        return {
            "symbol": symbol, "plan": plan, "log": log, "results": results, "criteria": criteria,
            "report": loop["final"], "history": loop["history"],
        }

    # Write {"score", "lesson"} from run()'s output; the lesson (or None) applies to any symbol. The
    # score and feedback are the writing loop's last grade of the final note, so every score shown
    # comes from one grading: in testing, a second Critic call on the same note disagreed with the
    # first (a pass, then a 4). The Critic grades the text; the Reflector reads the whole run: the
    # first draft's feedback says what to fix, and the note beside the data it was written from
    # shows what the Writer did with it.
    def reflect(self, symbol, run_result):
        final = run_result["history"][-1]
        record = {
            "symbol": symbol,
            "report": run_result["report"],
            "criteria_and_reference_data": run_result["criteria"],
            "plan": run_result["plan"],
            "tool_log": [{k: row[k] for k in ("tool", "ok", "result")} for row in run_result["log"]],
            "score_history": [{k: h[k] for k in ("round", "score", "pass")} for h in run_result["history"]],
            "first_draft_feedback": run_result["history"][0]["feedback"],
            "final_score": final["score"],
            "final_feedback": final["feedback"],
            "lessons_already_stored": self.lessons,
        }
        handoff("Critic", "Reflector", f"first draft {run_result['history'][0]['score']}/10, "
                                       f"{len(record['first_draft_feedback'])} items to learn from")
        reply = chat(REFLECTOR, json.dumps(record, indent=1), json_mode=True, agent="Reflector")
        lesson = reply.get("lesson") or None
        handoff("Reflector", "Coordinator", lesson or "nothing to store")
        return {"score": final["score"], "lesson": lesson}

    # Add a new lesson. The exact-match check only stops verbatim repeats; the Reflector is told not to
    # restate one.
    def remember(self, lesson):
        if lesson and lesson not in self.lessons:
            self.lessons.append(lesson)


### Agent Functions and Capabilities

Each function is a method of `ResearchAgent` in the cell above, and each is demonstrated in the output of the demonstration that follows.

**1. Plans (`plan`).** The Coordinator makes one JSON-mode call whose prompt holds the goal and `MENU`: what each step returns, what it costs in LLM calls, and its arguments. It returns at most eight steps, each with the tool, its arguments, and one sentence on why. *Shown by:* the plan table printed for each run, with the Coordinator's reason for every step. The plan rarely changes between runs, by design: every criterion needs prices, fundamentals, or the chain, and the Writer is told to take risks from the specialists, so with one goal the four-step plan is the natural one. We did not invent goals or steps to make the planning vary for show. The choice is still the model's, and earlier versions showed that it matters: plans that skipped `run_chain` produced notes that failed the sentiment criterion.

**2. Uses tools dynamically (`run`, `call_step`, `STEPS`).** The executor walks the plan and calls each step by name from the `STEPS` dispatch table: the price and fundamentals tools, `run_chain` (section 1), and `route_and_analyze` (section 2). No line of our code fixes which tools run or in what order. `route_and_analyze` receives the chain's labeled articles when the plan ran the chain, and the raw news otherwise. A step that fails, or names a tool that does not exist, is logged and skipped, so a run that loses one source still produces a note and the failure becomes evidence for reflection. *Shown by:* the `Coordinator → <tool>` handoff lines at the start of each run, each workflow's own handoffs, and the table of what each step returned.

**3. Self-reflects (`evaluator_optimizer` inside `run`, then `reflect`).** Reflection happens at two levels. Within a run, the note goes through section 3's loop: the Critic grades the draft against seven criteria and the data it was written from, and the Writer revises from the feedback, at most twice. After the run, the Reflector reads the whole record (the note, the reference data, the plan, the tool log, and the score history) and writes one lesson about what the first draft missed; it is told to store nothing if the first draft drew no feedback. It learns from the first draft because the loop usually revises a weak draft to a pass, and a Reflector that saw only the final grade had nothing to learn. *Shown by:* the Writer and Critic handoffs in each run, the Critic's feedback printed under each note, and the `Critic → Reflector` and `Reflector → Coordinator` lines carrying each first-draft grade and lesson.

**4. Learns across runs (`remember`, `recall`).** Memory is a list of lessons on the agent, empty when the agent is created, so every run of this notebook starts cold and the learning shown happened inside this execution. `remember` adds a new lesson, and the Writer reads every stored lesson when it drafts on later runs, for any symbol. Memory reaches the Writer alone: earlier versions also gave it to the planner, but once every plan was the same four steps, the planner had nothing to learn. Lessons are instructions, not figures, because the chain's labels vary between runs, and a lesson quoting the last run's count would contradict the next run's data. *Shown by:* the lessons each run's Writer read, the memory table after the last run, and the first-draft scores with and without memory.

**A criterion chosen so learning can show.** Criterion 7 (name the most common news topic, with its count) was added because a first draft rarely meets it unprompted and one lesson teaches it, so the demonstration can show learning. The Writer's drafting prompt never shows the criteria (it sees them only when revising), so a first draft meets criterion 7 only through memory. The claim this supports is that memory teaches standards the Writer is never told when drafting, not that memory improves notes in general.

### Evaluation and Iteration

How the demonstration measures learning:

- **Interleaved runs.** One agent runs four times: AAPL, NVDA, AAPL, NVDA. NVDA's first run reads only AAPL's lessons, so a difference there shows a lesson transferring between symbols. Each symbol's second run reads every lesson stored before it.
- **Twin runs.** Every run with lessons to read is paired with a twin: the same run by a new agent with empty memory. The two differ only in memory, so the twin is the control for the whole run. One pair is one sample, and at temperature 1.0 plans and grades vary between identical requests, so a twin suggests rather than proves.
- **Repeated trials.** The last cell is the evidence. For each run that had a twin, the first draft is written seven more times with that run's memory and seven times without, from the run's own stored data, and the Critic grades each draft once. Data, Writer, and Critic are the same on both sides; only memory differs.
- **First drafts, not final notes.** The writing loop revises most drafts to a pass, so final grades hide memory's effect. The figures plot the Critic's score of each first draft.

Every call samples at temperature 1.0, the model card's recommendation, so plans, wording, and scores differ between runs of this notebook; the twins and trials are what make one run readable. A run that fails (a truncated reply or a timeout) is retried once, and two failures skip the run without ending the demonstration.

The main changes testing led to (the full iteration log is in `docs/agent-walkthrough.md`):

| Observed in testing | Change |
|---|---|
| At temperature 0 the Critic sometimes reasoned inside its JSON until it hit the token cap, and temperature 0 was not repeatable on our gateway. | Every agent samples at 1.0; twins and trials, not determinism, attribute differences to memory. |
| A Critic asked for one 1-to-10 score gave the same text scores about two points apart. | Section 3's Critic judges each criterion separately, and the score is the share met. |
| A Reflector that saw only the final, usually passing, grade stored memory after 2 of 8 runs. | It reads the first draft's feedback: 6 of 8 runs stored memory, and the next first draft rose 1.17 points on average. |
| Memory written after clean runs, and rules about what to avoid, made later first drafts worse (by 0.28 points). | The Reflector is told to store nothing after a clean first draft and to say what to do; later first drafts then rose 0.63 points. |
| Lessons quoted the last run's figures, which the next run's data contradicted. | Lessons hold instructions, not figures. |
| Plans read the news twice, once through the news tool and once inside `run_chain`. | The news tool is not a planner step. |
| A five-article routing cap dropped AAPL's one article about a price-target cut. | Routing takes every article the chain keeps. |

### Demonstration

Four runs of one agent, each followed by its twin when it had lessons to read. Then every run's lessons, plan, tool results, and note; the memory after the last run; the first-draft scores and the work each agent did; and the repeated trials.

In [ ]:
# One agent, four runs, interleaved. NVDA's first run already reads the lessons from AAPL's, and each
# symbol's second run reads every lesson learned before it. Every run with lessons to read gets a
# twin: the same run by a new agent with empty memory.
# The two differ only in memory, so the twin is the control for the whole run: plan, data, note,
# and score.
demo_start = len(llm.CALL_LOG)
agent = ResearchAgent()
runs = []


# Call fn(symbol), retrying once. At temperature 1.0 a failed call (a truncated or malformed reply)
# is rare and random, so a second attempt usually succeeds; two failures skip the run instead of
# ending the demonstration. Retrying is safe because only remember() changes memory.
def with_retry(label, fn, symbol):
    for attempt in (1, 2):
        try:
            return fn(symbol)
        except Exception as error:  # noqa: BLE001  one failed call must not end the demonstration
            print(f"{label}, attempt {attempt} did not finish: {type(error).__name__}: {error}")
    return None


def run_and_reflect(symbol):
    result = agent.run(symbol)
    return result, agent.reflect(symbol, result)


def run_twin(symbol):
    return ResearchAgent().run(symbol)


for label, symbol in [("AAPL run 1", "AAPL"), ("NVDA run 1", "NVDA"), ("AAPL run 2", "AAPL"), ("NVDA run 2", "NVDA")]:
    print(f"\n=== {label}")
    lessons_before = list(agent.lessons)
    outcome = with_retry(label, run_and_reflect, symbol)
    if outcome is None:
        continue
    result, reflection = outcome
    agent.remember(reflection["lesson"])
    twin = None
    if lessons_before:
        print(f"\n--- {label}, twin: a new agent with empty memory")
        twin = with_retry(f"{label} twin", run_twin, symbol)
    runs.append({"label": label, "lessons_before": lessons_before, "twin": twin, **result})


In [ ]:
if runs:
    # Show a table left-aligned and in full, numbers to two decimals. The export is HTML, where pandas'
    # default right alignment makes sentences hard to read.
    def show(table):
        display(table.style.format(precision=2).set_properties(**{"text-align": "left"})
                .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

    # Number a table's rows from 1, the way a reader counts steps.
    def numbered(table, label="step"):
        table.index = range(1, len(table) + 1)
        return table.rename_axis(label)

    # A list of lessons as a numbered table.
    def lessons_table(lessons):
        return numbered(pd.DataFrame({"lesson": lessons}), label="lesson")

    # A research note, paragraph by paragraph, wrapped to the page, then what the Critic said about its
    # first draft. The first draft is what memory can change: the writing loop revises most drafts to a
    # pass, so final grades alone hide the difference.
    def print_note(heading, run):
        first, final = run["history"][0], run["history"][-1]
        print(f"\n{heading} (first draft {first['score']}/10, final {final['score']}/10):\n")
        for paragraph in run["report"].split("\n"):
            print(textwrap.fill(paragraph, 96) if paragraph.strip() else "")
        print("\nWhat the Critic said about the first draft:")
        for item in first["feedback"] or ["no failed criteria"]:
            print(textwrap.fill(f"- {item}", 96, subsequent_indent="  "))

    # Every run: the lessons its Writer read, its plan and why it chose each step, what each step
    # returned, and its note with its grades. A run with a twin also shows the twin's note; the only
    # difference between the two is memory.
    for run in runs:
        print(f"\n=== {run['label']}: the lessons its Writer read")
        if run["lessons_before"]:
            show(lessons_table(run["lessons_before"]))
        else:
            print("none yet: memory was empty")
        print("The plan, and why it chose each step:")
        show(numbered(pd.DataFrame(run["plan"], columns=["tool", "args", "why"])))
        print("What each step returned (first 80 characters):")
        show(numbered(pd.DataFrame(run["log"], columns=["tool", "ok", "result"])))
        if run["twin"] is None:
            print_note(run["label"], run)
        else:
            print_note(f"{run['label']} with memory", run)
            print_note(f"{run['label']} without memory (twin)", run["twin"])


In [ ]:
if runs:
    # Everything the agent learned across the runs.
    show(lessons_table(agent.lessons))


In [ ]:
if runs:
    # The Critic's grade of each first draft, for the learning agent and for each run's memoryless twin.
    # First drafts, because the writing loop revises most drafts to a pass and memory's effect is on what
    # the Writer gets right before any revision. The first run has no twin: its memory was empty, so it
    # was its own baseline. Below: how much work each agent did across the whole demonstration.
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.plot([run["label"] for run in runs], [run["history"][0]["score"] for run in runs],
            marker="o", label="with memory")
    twins = [run for run in runs if run["twin"]]
    ax.plot([run["label"] for run in twins], [run["twin"]["history"][0]["score"] for run in twins],
            marker="s", linestyle="--", label="without memory (twin)")
    ax.set(title="Critic score of the first draft, per run", ylabel="score", ylim=(0, 10.5))
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

    calls = llm.CALL_LOG[demo_start:]
    agents = sorted({c["agent"] for c in calls}, key=lambda name: -sum(c["agent"] == name for c in calls))
    count = [sum(c["agent"] == name for c in calls) for name in agents]
    tokens = [sum((c["prompt_tokens"] or 0) + (c["completion_tokens"] or 0) for c in calls if c["agent"] == name)
              for name in agents]
    fig, (left, right) = plt.subplots(1, 2, figsize=(9, 2.8), sharey=True)
    left.barh(agents, count)
    left.set(title="LLM calls per agent", xlabel="calls")
    left.invert_yaxis()
    right.barh(agents, tokens, color="tab:orange")
    right.set(title="Tokens per agent", xlabel="prompt + completion tokens")
    fig.tight_layout()
    plt.show()
    print(f"{len(calls)} calls, {sum(tokens):,} tokens, {sum(c['seconds'] for c in calls):.0f} s of model time")


In [ ]:
if runs:
    # Repeated trials. Each twin pair above is one draft per side, so a single lucky or unlucky draft
    # decides it. Here every run that had a twin has its first draft written TRIALS more times with the
    # memory its Writer read and TRIALS times with empty memory, from the run's own stored data, and the
    # Critic grades each draft once. Data, Writer, and Critic are the same in both rows; only memory
    # differs. Unlike a twin, which plans and labels the news again, nothing else can vary.
    TRIALS = 7
    trial_start = len(llm.CALL_LOG)
    rows = []
    for run in runs:
        if run["twin"] is None:
            continue
        remembering = ResearchAgent()
        remembering.lessons = run["lessons_before"]
        for condition, recall in [("with memory", remembering.recall()), ("without memory", ResearchAgent().recall())]:
            for _ in range(TRIALS):
                try:
                    grade = evaluate(write_report(run["symbol"], run["results"], recall), run["criteria"])
                except Exception as error:  # noqa: BLE001  a failed draft is skipped and counted, not fatal
                    print(f"{run['label']}, {condition}: draft skipped ({type(error).__name__})")
                    continue
                rows.append({"run": run["label"], "memory": condition, "mean score": grade["score"],
                             "all criteria met": grade["pass"],
                             **{f"missed criterion {c['number']}": not c["met"] for c in grade["checks"]}})

    # Means over the drafts: the score out of 10, and for every yes/no column the share of drafts (0 to 1).
    # No rows when no run had lessons to read (every first draft before the last run was clean).
    if not rows:
        print("No first drafts to compare: no run had lessons to read.")
    else:
        trials = pd.DataFrame(rows)
        print(f"First drafts per condition: {trials['memory'].value_counts().to_dict()}")
        summary = trials.groupby("memory", sort=False).mean(numeric_only=True).T
        show(summary)
        print("Share of first drafts meeting every criterion, per run:")
        show(trials.pivot_table(index="run", columns="memory", values="all criteria met", sort=False))
    trial_calls = llm.CALL_LOG[trial_start:]
    print(f"{len(trial_calls)} calls, {sum(c['seconds'] for c in trial_calls):.0f} s of model time")

#### What this run shows

*[FILL after the final run, from the outputs above: which run stored a lesson and what it said; whether it reached the other symbol; the twin comparisons (first-draft scores with and without memory); the trials' mean scores and the criteria memory moved; any failures or retries.]*

## 5. Discussion and limitations

**What worked.** The four sections were built in parallel against written contracts (function names, arguments, and return shapes fixed in the first week), and each workflow dropped into the agent without changes to the agent's code. The agent shows all four required functions in one execution, and the twin runs and repeated trials measure learning inside the notebook instead of asserting it. *[FILL after the final run: the measured effect, for example the mean first-draft score with and without memory, and the criteria it moved.]*

**What the evaluator catches and misses.** The Critic reliably catches missing content and invented figures: in testing, it failed every note that invented sentiment counts. It is still an LLM grading an LLM. Its score for one unchanged note varied with a standard deviation of about 1.5 points, and its judgment of risks is its least consistent. In one test run it passed a note that cited an 83 percent dot-com loss as an NVDA risk, a figure from an article about the Nasdaq-100 that the chain's classifier had kept as relevant to NVDA.

**What memory changed.** Memory taught the Writer standards it is never told when drafting: stating sentiment as three counts (criterion 3) and naming the most common topic with its count (criterion 7, chosen so that learning could show, as disclosed in section 4). It did not help with the criterion that asks for judgment: in testing, a supported risk (criterion 4) was missed as often with memory as without.

**Limitations.**

- Two symbols and one model. The runs show the mechanism working; evidence that memory improves research notes in general would need many symbols and repeated runs.
- Grades come from an LLM Critic that we have not checked against human grading.
- Every call samples at temperature 1.0, so a rerun gives different plans, labels, notes, and scores. The chain's classifier, for example, keeps a different set of articles on identical data.
- The data are a Yahoo Finance snapshot cached on September 19, 2026 (prices through September 18), so the notes describe the market as of that date.
- The planner has one goal, so its plan rarely changes. A planner with a real choice would need goals that need different data, such as a valuation-only note graded on its own criteria.

**Future work.**

- Send a failed criterion back to the specialist that owns the data (a missing earnings figure to the Earnings Specialist) before the Writer revises.
- Let the Coordinator replan when a step fails, instead of logging the failure and moving on.
- Keep memory across sessions, with pruning so the Writer's prompt does not grow without bound.
- Add SEC EDGAR filings or FRED economic series as tools: one cached function, one line in `STEPS`, one entry in `MENU`.
- Grade a sample of notes by hand and measure how often the Critic agrees.

## 6. AI-use disclosure and references

Course policy permits generative-AI tools with disclosure. Each of us logged our use, with what we verified or changed afterward, in [`docs/ai-use-log.md`](https://github.com/nazeshinjite/not-financial-advice/blob/main/docs/ai-use-log.md):

- **Jackson Kenyon (section 1):** GitHub Copilot in VS Code (GitHub, 2026) helped implement the prompt chain, refine the classification prompt and its `other` filter, and add the classifier's retry.
- **Ali Abdul-Hameed (section 2):** ChatGPT (OpenAI, 2026a) gave code review and feedback on the routing workflow and its follow-up fixes.
- **Nina Zhao (section 3):** Claude (Anthropic, 2026a) drafted the evaluator-optimizer notebook to the team's contract and a later cleanup.
- **Ian Schmitt (section 4 and assembly):** Claude Code (Anthropic, 2026b) drafted `src/llm.py`, `src/tools.py`, the agent, the documentation, and the assembly of this notebook, and ran the memory and sampling experiments; Codex CLI (OpenAI, 2026b) ran an adversarial review of the agent notebook.

Each of us ran our own section before merging it; the log records what each person verified.

### References

Anthropic. (2026a). *Claude* [Large language model]. https://claude.ai [VERIFY: model and version used for section 3]

Anthropic. (2026b). *Claude Code* (Claude Fable 5.1 and Claude Opus 5.5) [Computer software]. https://claude.com/claude-code [VERIFY: URL]

Aroussi, R. (n.d.). *yfinance* (Version 1.7.0) [Computer software]. GitHub. https://github.com/ranaroussi/yfinance

DeepSeek-AI. (2026). *DeepSeek V4.1 Flash* [Large language model]. Accessed through Nous Portal. [VERIFY: model card URL and release date]

GitHub. (2026). *GitHub Copilot* [Computer software]. https://github.com/features/copilot

Honnibal, M., Montani, I., Van Landeghem, S., & Boyd, A. (2020). *spaCy: Industrial-strength natural language processing in Python* [Computer software]. Zenodo. https://doi.org/10.5281/zenodo.1212303

OpenAI. (2026a). *ChatGPT* [Large language model]. https://chatgpt.com [VERIFY: version used for section 2]

OpenAI. (2026b). *Codex CLI* (GPT-6 Astra) [Computer software]. https://github.com/openai/codex [VERIFY: URL]

Schluntz, E., & Zhang, B. (2024, December 19). *Building effective agents*. Anthropic. https://www.anthropic.com/engineering/building-effective-agents [VERIFY: date]